# 09 - Your Own Library

This microlesson is about taking the code from the last lesson that was cobbled together, and trying to organize it into a reusable set of functions and a class. This could be something we place in the src folder then include it to be used. 

In [1]:
import json
from pathlib import Path
from IPython.display import display
from ipyleaflet import Map, Marker, LayersControl, WidgetControl
import ipywidgets as widgets


class GeoJsonHelp:
    """
    Small helper class for loading a GeoJSON file and pulling out pieces of it.
    """

    def __init__(self, path: Path = None):
        self.geojson_obj = None
        self.path = path
        if self.path:
            self.load_geojson(self.path)

    def load_geojson(self, path: Path):
        if not path.exists():
            raise FileNotFoundError(
                f"\n❌ ERROR: File not found:\n{path}\n"
                "Check spelling and folder structure."
            )

        self.geojson_obj = json.loads(path.read_text())
        return self.geojson_obj

    def get_features(self, geojson_obj):
        return geojson_obj.get("features", [])

    def extract_polygon_rings(self, feature):
        geom = feature["geometry"]
        coords = geom["coordinates"]

        if geom["type"] == "Polygon":
            return coords

        if geom["type"] == "MultiPolygon":
            rings = []
            for poly in coords:
                rings.extend(poly)
            return rings

        return []


class MapAppHelp:
    def __init__(self, points_file: Path, center=(40.0, -99.0), zoom=5):
        self.points_file = points_file
        self.clicked_points = self.load_points()
        self.markers = []

        self.map = Map(
            center=center,
            zoom=zoom,
            layout=widgets.Layout(width="100%", height="700px"),
        )
        self.map.add(LayersControl())

        self.output = widgets.Output()
        self.clear_btn = widgets.Button(description="Clear Saved Points")

        self.restore_markers()
        self.bind_events()
        self.add_controls()

    def load_points(self):
        if self.points_file.exists():
            text = self.points_file.read_text().strip()
            if text:
                return json.loads(text)
            return []

        self.points_file.parent.mkdir(parents=True, exist_ok=True)
        self.points_file.write_text("[]")
        return []

    def save_points(self):
        self.points_file.parent.mkdir(parents=True, exist_ok=True)
        self.points_file.write_text(json.dumps(self.clicked_points, indent=2))

    def restore_markers(self):
        for pt in self.clicked_points:
            marker = Marker(location=(pt["lat"], pt["lon"]))
            self.markers.append(marker)
            self.map.add(marker)

    def add_marker(self, lat: float, lon: float):
        marker = Marker(location=(lat, lon))
        self.markers.append(marker)
        self.map.add(marker)

    def log(self, message: str):
        with self.output:
            print(message)

    def handle_interaction(self, **kwargs):
        if kwargs.get("type") != "click":
            return

        lat, lon = kwargs["coordinates"]
        point = {"lat": round(lat, 6), "lon": round(lon, 6)}

        self.clicked_points.append(point)
        self.add_marker(point["lat"], point["lon"])
        self.save_points()
        self.log(f"Saved click: {point}")

    def clear_points(self, _):
        self.clicked_points.clear()
        self.save_points()

        for marker in list(self.markers):
            if marker in self.map.layers:
                self.map.remove(marker)

        self.markers.clear()
        self.log("Cleared saved points.")

    def bind_events(self):
        self.map.on_interaction(self.handle_interaction)
        self.clear_btn.on_click(self.clear_points)

    def add_controls(self):
        self.map.add(WidgetControl(widget=self.clear_btn, position="topright"))

    def display(self):
        display(self.map, self.output)


## Your Turn

Take the code from above and place it in a file in the `src/wdo` folder called `mapAppHelp.py` then use the appropriate `from wdo.mapAppHelp import ....` commands in order to  recreate the map and interaction above.

In [2]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").exists()), None)
assert repo_root is not None, "❌ Could not locate the project root."

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from wdo.mapAppHelp import GeoJsonHelp, MapAppHelp

countries_path = repo_root / "data" / "countries.geojson"
points_file = repo_root / "data" / "clicked_points.json"

geo_helper = GeoJsonHelp(countries_path)
feature_count = len(geo_helper.get_features(geo_helper.geojson_obj))
print(f"Loaded {feature_count} country features from {countries_path.name}.")

app = MapAppHelp(points_file=points_file, center=(33.87372, -98.51947), zoom=5)
app.display()


Loaded 255 country features from countries.geojson.


Map(center=[33.87372, -98.51947], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', …

Output()